In [3]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder

from gensim.models import Word2Vec

import torch
from transformers import AutoTokenizer, AutoModel

Load Dataset

In [4]:
import glob
import pandas as pd

raw_files = (
    glob.glob('dataset/**/RAW/*.xlsx', recursive=True) +
    glob.glob('dataset/**/RAW/*.csv', recursive=True)
)
print('RAW files found:', raw_files)

dfs = []
for f in raw_files:
    if f.endswith('.xlsx'):
        df = pd.read_excel(f)
    else:
        df = pd.read_csv(f)
    print(f'{f} -> shape: {df.shape}, columns: {df.columns.tolist()}')
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print('\nTotal shape:', data.shape)
print(data.head(3))
print('\nColumns:', data.columns.tolist())

RAW files found: ['dataset\\RAW\\dataset_cnn_10k.xlsx', 'dataset\\RAW\\dataset_kompas_4k.xlsx', 'dataset\\RAW\\dataset_tempo_6k.xlsx', 'dataset\\RAW\\dataset_turnbackhoax_10k.xlsx']
dataset\RAW\dataset_cnn_10k.xlsx -> shape: (10000, 6), columns: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']
dataset\RAW\dataset_kompas_4k.xlsx -> shape: (4750, 6), columns: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']
dataset\RAW\dataset_tempo_6k.xlsx -> shape: (6592, 6), columns: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']
dataset\RAW\dataset_turnbackhoax_10k.xlsx -> shape: (10384, 6), columns: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']

Total shape: (31726, 6)
                                               Title  \
0  Anies di Milad BKMT: Pengajian Menghasilkan Ib...   
1  Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...   
2  PKB Bakal Daftarkan Menaker Ida Fauziyah Jadi ...   

                       Timestamp  \
0  Selasa, 21 Feb 2023 21:

Data Cleaning & Label Encoding

In [5]:
print("Semua kolom:", data.columns.tolist())
print(data.head(2))

Semua kolom: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']
                                               Title  \
0  Anies di Milad BKMT: Pengajian Menghasilkan Ib...   
1  Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...   

                       Timestamp  \
0  Selasa, 21 Feb 2023 21:22 WIB   
1  Selasa, 21 Feb 2023 20:46 WIB   

                                            FullText  \
0  Jakarta, CNN Indonesia -- Mantan Gubernur DKI ...   
1  Medan, CNN Indonesia -- Gubernur Sumatera Utar...   

                                                Tags         Author  \
0  anies baswedan;pengajian;pilpres 2024;badan ko...  CNN Indonesia   
1             edy rahmayadi;pemilu 2024;pilkada 2024  CNN Indonesia   

                                                 Url  
0  https://www.cnnindonesia.com/nasional/20230221...  
1  https://www.cnnindonesia.com/nasional/20230221...  


In [6]:
# Cell 4 - Load & beri label per source
import glob
import pandas as pd

hoax_files = (
    glob.glob('dataset/**/RAW/dataset_turnbackhoax*.xlsx', recursive=True) +
    glob.glob('dataset/**/RAW/dataset_turnbackhoax*.csv', recursive=True)
)
real_files = (
    glob.glob('dataset/**/RAW/dataset_cnn*.xlsx', recursive=True) +
    glob.glob('dataset/**/RAW/dataset_kompas*.xlsx', recursive=True) +
    glob.glob('dataset/**/RAW/dataset_tempo*.xlsx', recursive=True) +
    glob.glob('dataset/**/RAW/dataset_cnn*.csv', recursive=True) +
    glob.glob('dataset/**/RAW/dataset_kompas*.csv', recursive=True) +
    glob.glob('dataset/**/RAW/dataset_tempo*.csv', recursive=True)
)

print('Hoax files:', hoax_files)
print('Real files:', real_files)

def load_files(file_list):
    dfs = []
    for f in file_list:
        df = pd.read_excel(f) if f.endswith('.xlsx') else pd.read_csv(f)
        print(f'{f} -> {df.shape} | cols: {df.columns.tolist()}')
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

df_hoax = load_files(hoax_files)
df_real = load_files(real_files)

print('\nHoax shape:', df_hoax.shape)
print('Real shape:', df_real.shape)
print('\nHoax columns:', df_hoax.columns.tolist())
print('Real columns:', df_real.columns.tolist())

Hoax files: ['dataset\\RAW\\dataset_turnbackhoax_10k.xlsx']
Real files: ['dataset\\RAW\\dataset_cnn_10k.xlsx', 'dataset\\RAW\\dataset_kompas_4k.xlsx', 'dataset\\RAW\\dataset_tempo_6k.xlsx']
dataset\RAW\dataset_turnbackhoax_10k.xlsx -> (10384, 6) | cols: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']
dataset\RAW\dataset_cnn_10k.xlsx -> (10000, 6) | cols: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']
dataset\RAW\dataset_kompas_4k.xlsx -> (4750, 6) | cols: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']
dataset\RAW\dataset_tempo_6k.xlsx -> (6592, 6) | cols: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']

Hoax shape: (10384, 6)
Real shape: (21342, 6)

Hoax columns: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']
Real columns: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']


In [7]:
# Cell 5
HOAX_TEXT_COL = 'FullText'
REAL_TEXT_COL = 'FullText'

df_hoax_clean = pd.DataFrame({
    'text': df_hoax[HOAX_TEXT_COL].astype(str),
    'label': 'hoax'
})

df_real_clean = pd.DataFrame({
    'text': df_real[REAL_TEXT_COL].astype(str),
    'label': 'real'
})

data = pd.concat([df_hoax_clean, df_real_clean], ignore_index=True)
data = data.dropna(subset=['text'])
data = data.drop_duplicates(subset=['text'])
data = data[data['text'].str.strip() != '']

le = LabelEncoder()
data['label_enc'] = le.fit_transform(data['label'])

TEXT_COL = 'text'
LABEL_COL = 'label'

print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))
print('Label distribution:')
print(data['label'].value_counts())
print('\nTotal data:', len(data))

Label mapping: {'hoax': 0, 'real': 1}
Label distribution:
label
real    20944
hoax    10367
Name: count, dtype: int64

Total data: 31311


Preprocessing & Function

In [8]:
factory_stemmer = StemmerFactory()
stemmer = factory_stemmer.create_stemmer()

factory_stopword = StopWordRemoverFactory()
stopword_list = set(factory_stopword.get_stop_words())

def case_folding(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return text.split()

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stopword_list]

def apply_stemming(tokens):
    return [stemmer.stem(t) for t in tokens]

def preprocess(text, do_stopword=False, do_stemming=False):
    # C0 baseline: case folding + tokenization always applied
    text = case_folding(text)
    tokens = tokenize(text)
    if do_stopword:
        tokens = remove_stopwords(tokens)
    if do_stemming:
        tokens = apply_stemming(tokens)
    return ' '.join(tokens)

In [9]:
from tqdm.notebook import tqdm
tqdm.pandas()

# Class imbalance ditangani oleh class_weight='balanced' di SVM
print('Total data (full dataset):', len(data))
print(data['label'].value_counts())
print('\nClass ratio (real:hoax):', round(data['label'].value_counts()['real'] / data['label'].value_counts()['hoax'], 1), ':1')


Total data (full dataset): 31311
label
real    20944
hoax    10367
Name: count, dtype: int64

Class ratio (real:hoax): 2.0 :1


In [10]:
configs = {
    'C0_baseline':      {'do_stopword': False, 'do_stemming': False},
    'C1_stopword':      {'do_stopword': True,  'do_stemming': False},
    'C2_stemming':      {'do_stopword': False, 'do_stemming': True},
    'C3_full_pipeline': {'do_stopword': True,  'do_stemming': True},
}

# Preprocessing pada full dataset (~23,179 artikel)
preprocessed = {}
for cfg_name, cfg_args in configs.items():
    print(f'\nProcessing {cfg_name}...')
    preprocessed[cfg_name] = data[TEXT_COL].progress_apply(
        lambda x: preprocess(x, **cfg_args)
    )

print('\nDone semua konfigurasi')



Processing C0_baseline...


  0%|          | 0/31311 [00:00<?, ?it/s]


Processing C1_stopword...


  0%|          | 0/31311 [00:00<?, ?it/s]


Processing C2_stemming...


  0%|          | 0/31311 [00:00<?, ?it/s]


Processing C3_full_pipeline...


  0%|          | 0/31311 [00:00<?, ?it/s]


Done semua konfigurasi


In [11]:
import os, pickle

CACHE_FILE = 'preprocessed_cache_full.pkl'

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'rb') as f:
        preprocessed = pickle.load(f)
    print('Loaded from cache (full dataset)')
else:
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(preprocessed, f)
    print('Saved to cache (full dataset)')

Loaded from cache (full dataset)


Train/Test Split (80/20)

In [14]:
y = data['label_enc'].values

splits = {}
for cfg_name in configs:
    X = preprocessed[cfg_name].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    splits[cfg_name] = (X_train, X_test, y_train, y_test)

Evaluation Helper

In [15]:
def evaluate(y_true, y_pred):
    return {
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 4),
        'F1-Score':  round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 4),
    }

Experiment 1 — TF-IDF + SVM

In [16]:
tfidf_results = {}

for cfg_name in configs:
    X_train, X_test, y_train, y_test = splits[cfg_name]

    vectorizer = TfidfVectorizer()
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec  = vectorizer.transform(X_test)

    svm = SVC(kernel='linear', class_weight='balanced', random_state=42)
    svm.fit(X_train_vec, y_train)
    y_pred = svm.predict(X_test_vec)

    tfidf_results[cfg_name] = evaluate(y_test, y_pred)
    print(f'{cfg_name}: {tfidf_results[cfg_name]}')

df_tfidf = pd.DataFrame(tfidf_results).T
print('\n=== TF-IDF Results ===')
print(df_tfidf.to_string())

C0_baseline: {'Accuracy': 0.997, 'Precision': 0.997, 'Recall': 0.997, 'F1-Score': 0.997}
C1_stopword: {'Accuracy': 0.9978, 'Precision': 0.9978, 'Recall': 0.9978, 'F1-Score': 0.9978}
C2_stemming: {'Accuracy': 0.9971, 'Precision': 0.9971, 'Recall': 0.9971, 'F1-Score': 0.9971}
C3_full_pipeline: {'Accuracy': 0.9974, 'Precision': 0.9974, 'Recall': 0.9974, 'F1-Score': 0.9974}

=== TF-IDF Results ===
                  Accuracy  Precision  Recall  F1-Score
C0_baseline         0.9970     0.9970  0.9970    0.9970
C1_stopword         0.9978     0.9978  0.9978    0.9978
C2_stemming         0.9971     0.9971  0.9971    0.9971
C3_full_pipeline    0.9974     0.9974  0.9974    0.9974


Experiment 2 — Word2Vec + SVM

In [19]:
def doc_vector(tokens_list, model, vector_size):
    vectors = []
    for tokens in tokens_list:
        t = tokens.split() if isinstance(tokens, str) else tokens
        vecs = [model.wv[w] for w in t if w in model.wv]
        if vecs:
            vectors.append(np.mean(vecs, axis=0))
        else:
            vectors.append(np.zeros(vector_size))
    return np.array(vectors)

w2v_results = {}
VECTOR_SIZE = 100

for cfg_name in configs:
    X_train, X_test, y_train, y_test = splits[cfg_name]

    train_tokens = [t.split() for t in X_train]
    w2v_model = Word2Vec(sentences=train_tokens, vector_size=VECTOR_SIZE,
                         window=5, min_count=1, workers=4, seed=42)

    X_train_vec = doc_vector(X_train, w2v_model, VECTOR_SIZE)
    X_test_vec  = doc_vector(X_test,  w2v_model, VECTOR_SIZE)

    svm = SVC(kernel='linear', class_weight='balanced', random_state=42)
    svm.fit(X_train_vec, y_train)
    y_pred = svm.predict(X_test_vec)

    w2v_results[cfg_name] = evaluate(y_test, y_pred)
    print(f'{cfg_name}: {w2v_results[cfg_name]}')

df_w2v = pd.DataFrame(w2v_results).T
print('\n=== Word2Vec Results ===')
print(df_w2v.to_string())

C0_baseline: {'Accuracy': 0.9943, 'Precision': 0.9943, 'Recall': 0.9943, 'F1-Score': 0.9943}
C1_stopword: {'Accuracy': 0.9957, 'Precision': 0.9957, 'Recall': 0.9957, 'F1-Score': 0.9957}
C2_stemming: {'Accuracy': 0.9943, 'Precision': 0.9943, 'Recall': 0.9943, 'F1-Score': 0.9943}
C3_full_pipeline: {'Accuracy': 0.9958, 'Precision': 0.9959, 'Recall': 0.9958, 'F1-Score': 0.9959}

=== Word2Vec Results ===
                  Accuracy  Precision  Recall  F1-Score
C0_baseline         0.9943     0.9943  0.9943    0.9943
C1_stopword         0.9957     0.9957  0.9957    0.9957
C2_stemming         0.9943     0.9943  0.9943    0.9943
C3_full_pipeline    0.9958     0.9959  0.9958    0.9959


Experiment 3 — IndoBERT + SVM

In [21]:
INDOBERT_MODEL = 'indobenchmark/indobert-base-p1'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

tokenizer_bert = AutoTokenizer.from_pretrained(INDOBERT_MODEL)
bert_model = AutoModel.from_pretrained(INDOBERT_MODEL).to(device)
bert_model.eval()

def get_cls_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        encoded = tokenizer_bert(
            batch, padding=True, truncation=True,
            max_length=512, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            output = bert_model(**encoded)
        cls = output.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls)
        if (i // batch_size) % 10 == 0:
            print(f'  batch {i // batch_size + 1}/{(len(texts) + batch_size - 1) // batch_size}')
    return np.vstack(all_embeddings)

print('IndoBERT loaded')

Device: cpu
IndoBERT loaded


In [22]:
import gc
import torch

def get_cls_embeddings(texts, batch_size=8):  # turunin dari default ke 8
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, 
                           max_length=128, return_tensors='pt')
        with torch.no_grad():
            output = model(**encoded)
        embeddings = output.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)
        
        # Bersihkan memori tiap batch
        del encoded, output, embeddings
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    return np.vstack(all_embeddings)

In [24]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")
model.eval()

# Pindah ke CPU aja kalau RAM GPU terbatas
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [26]:
def get_cls_embeddings(texts, batch_size=8):
    # Pastikan semua input adalah string
    texts = [str(t) if t is not None else '' for t in texts]
    
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, 
                           max_length=128, return_tensors='pt')
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            output = model(**encoded)
        embeddings = output.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)
        
        del encoded, output, embeddings
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    return np.vstack(all_embeddings)

In [27]:
cfg_name = 'C0_baseline'
X_train, X_test, y_train, y_test = splits[cfg_name]
X_train_vec = get_cls_embeddings(X_train)

In [30]:
indobert_results = {}

for cfg_name in configs:
    X_train, X_test, y_train, y_test = splits[cfg_name]
    
    X_train_vec = get_cls_embeddings(list(X_train))
    X_test_vec  = get_cls_embeddings(list(X_test))
    
    svm = SVC(kernel='linear', class_weight='balanced', random_state=42)
    svm.fit(X_train_vec, y_train)
    y_pred = svm.predict(X_test_vec)
    
    indobert_results[cfg_name] = evaluate(y_test, y_pred)
    print(f'{cfg_name}: {indobert_results[cfg_name]}')

C0_baseline: {'Accuracy': 0.9958, 'Precision': 0.9958, 'Recall': 0.9958, 'F1-Score': 0.9958}
C1_stopword: {'Accuracy': 0.9958, 'Precision': 0.9959, 'Recall': 0.9958, 'F1-Score': 0.9958}
C2_stemming: {'Accuracy': 0.9951, 'Precision': 0.995, 'Recall': 0.9951, 'F1-Score': 0.995}
C3_full_pipeline: {'Accuracy': 0.9955, 'Precision': 0.9955, 'Recall': 0.9955, 'F1-Score': 0.9955}


In [31]:
def build_summary(results, name):
    rows = []
    for cfg, metrics in results.items():
        row = {'Embedding': name, 'Config': cfg}
        row.update(metrics)
        rows.append(row)
    return pd.DataFrame(rows)

summary = pd.concat([
    build_summary(tfidf_results,    'TF-IDF'),
    build_summary(w2v_results,      'Word2Vec'),
    build_summary(indobert_results, 'IndoBERT'),
], ignore_index=True).set_index(['Embedding', 'Config'])

print(summary.to_string())
summary.to_csv('ablation_results.csv')
print('\nSaved to ablation_results.csv')

                            Accuracy  Precision  Recall  F1-Score
Embedding Config                                                 
TF-IDF    C0_baseline         0.9970     0.9970  0.9970    0.9970
          C1_stopword         0.9978     0.9978  0.9978    0.9978
          C2_stemming         0.9971     0.9971  0.9971    0.9971
          C3_full_pipeline    0.9974     0.9974  0.9974    0.9974
Word2Vec  C0_baseline         0.9943     0.9943  0.9943    0.9943
          C1_stopword         0.9957     0.9957  0.9957    0.9957
          C2_stemming         0.9943     0.9943  0.9943    0.9943
          C3_full_pipeline    0.9958     0.9959  0.9958    0.9959
IndoBERT  C0_baseline         0.9958     0.9958  0.9958    0.9958
          C1_stopword         0.9958     0.9959  0.9958    0.9958
          C2_stemming         0.9951     0.9950  0.9951    0.9950
          C3_full_pipeline    0.9955     0.9955  0.9955    0.9955

Saved to ablation_results.csv
